# 🧠 最小 AI 编程智能体 — 从零理解 Agent Loop

> **目标**：用约 150 行 Python 代码，从零搭建一个 AI 编程助手，理解大模型「智能体」的工作原理。

---

## 你将学到什么？

- 大模型（LLM）如何通过 **函数调用（Function Calling）** 操作你的文件
- **Agent Loop** 是什么——整个 AI 编程助手的核心架构
- 亲手实现一个能读文件、写文件、列出目录的 AI 助手

## 你需要准备什么？

| 你需要 | 说明 |
|--------|------|
| LLM API Key | 任一支持 OpenAI 兼容接口的模型均可（智谱 / DeepSeek / Qwen / OpenAI 等）|
| Python 基础 | 了解变量、函数、`print()` 即可 |

> 💡 免费额度推荐：[智谱 GLM-4](https://open.bigmodel.cn/) / [DeepSeek](https://platform.deepseek.com/)

## 学习路径

1. **Part 1** ← 你在这里：安装依赖，配置模型，验证连接
2. **Part 2**：直接体验 Agent 的效果（先感受，不用管代码）
3. **Part 3**：逐段拆解代码，理解每一个部分
4. **Part 4**：🎉 自由探索你的 Agent

> ⚠️ **请按顺序执行每个 Cell**，不要跳着运行。


## Part 1：环境准备

### 1.1 安装依赖

In [ ]:
!pip install openai -q

### 1.2 配置你的模型

> ⚠️ **重要提醒**：API Key 相当于你账户的密码。**不要**把含 Key 的 Notebook 分享给他人！
> 每次重新打开 Notebook 需要重新填入 Key。

选择任一支持 OpenAI 兼容接口的模型，修改下方配置后运行。

| Provider | base_url | 免费额度 |
|----------|----------|---------|
| 智谱 | `https://open.bigmodel.cn/api/paas/v4` | ✅ 有 |
| DeepSeek | `https://api.deepseek.com` | ✅ 有 |
| Qwen（阿里） | `https://dashscope.aliyuncs.com/compatible-mode/v1` | ✅ 有 |
| OpenAI | `https://api.openai.com/v1` | ❌ 付费 |
| 其他兼容接口 | 问你用的提供商 | - |

In [1]:
# 👇 改下面三行为你自己的配置
API_KEY = "********************************"
BASE_URL = "https://open.bigmodel.cn/api/paas/v4"   # 改成你用的服务商地址
MODEL = "glm-4.5-air"                                # 改成你用的模型名

print("✅ 配置已保存（请运行下一个 Cell 验证连接）")

✅ 配置已保存（请运行下一个 Cell 验证连接）


### 1.3 验证模型连接

发送一条测试消息，确认 API 可用。

In [2]:
from openai import OpenAI

client = OpenAI(api_key=API_KEY, base_url=BASE_URL)

try:
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": "你好，请回复'连接成功'"}],
        max_tokens=50
    )
    msg = response.choices[0].message
    # 检查 content 是否为 None（某些 provider 在 function calling 模式下不返回 content）
    reply = msg.content
    if reply is None or reply.strip() == "":
        print(f"⚠️  模型返回 content 为空（{type(reply)}），但 API 调用本身成功")
        print(f"   模型信息: {response.model}")
        print(f"   这通常不影响后续 function calling 功能")
    else:
        print(f"✅ 连接成功！模型回复：{reply}")
        print(f"   使用模型：{response.model}")
except Exception as e:
    print(f"❌ 连接失败：{e}")
    print(f"   请检查：")
    print(f"     1) API Key 是否正确")
    print(f"     2) BASE_URL 是否正确（注意末尾不要带 /chat/completions）")
    print(f"     3) MODEL 名称是否正确")

⚠️  模型返回 content 为空（<class 'str'>），但 API 调用本身成功
   模型信息: glm-4.5-air
   这通常不影响后续 function calling 功能


---

## Part 2：先感受！Agent 能做什么

> 下面这个 Cell 定义了完整的智能体。**先点运行，不用管里面是什么代码**——我们会在 Part 3 逐行拆解。
> 
> 运行成功后，你会看到 Agent 已经准备好，等待你的指令。

In [3]:
import json
import os
from pathlib import Path

# ═══════════════════════════════════════════
# Agent 完整定义（点击左侧三角展开查看）
# ═══════════════════════════════════════════

# 路径工具
def resolve_path(path_str: str) -> Path:
    """相对路径转绝对路径"""
    p = Path(path_str).expanduser()
    return p.resolve() if not p.is_absolute() else p

# 工具 1：读取文件
def read_file(path: str) -> dict:
    """读取文件的完整内容"""
    full_path = resolve_path(path)
    content = full_path.read_text(encoding="utf-8")
    return {"file_path": str(full_path), "content": content}

# 工具 2：列出目录
def list_files(path: str) -> dict:
    """列出目录中的文件和子目录"""
    full_path = resolve_path(path)
    items = []
    for item in sorted(full_path.iterdir()):
        items.append({"filename": item.name, "type": "file" if item.is_file() else "dir"})
    return {"path": str(full_path), "files": items}

# 工具 3：编辑文件
def edit_file(path: str, old_str: str, new_str: str) -> dict:
    """编辑文件：old_str为空则创建，非空则替换首次出现"""
    full_path = resolve_path(path)
    if old_str == "":
        full_path.write_text(new_str, encoding="utf-8")
        return {"path": str(full_path), "action": "created_file"}
    original = full_path.read_text(encoding="utf-8")
    if old_str not in original:
        return {"path": str(full_path), "action": "old_str not found"}
    edited = original.replace(old_str, new_str, 1)
    full_path.write_text(edited, encoding="utf-8")
    return {"path": str(full_path), "action": "edited"}

# 工具注册表
TOOL_REGISTRY = {
    "read_file": read_file,
    "list_files": list_files,
    "edit_file": edit_file,
}

# 工具 JSON Schema（LLM 的「菜单」）
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "read_file",
            "description": "读取文件的完整内容",
            "parameters": {
                "type": "object",
                "properties": {"path": {"type": "string", "description": "文件路径"}},
                "required": ["path"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "list_files",
            "description": "列出目录中的文件和子目录",
            "parameters": {
                "type": "object",
                "properties": {"path": {"type": "string", "description": "目录路径"}},
                "required": ["path"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "edit_file",
            "description": "编辑文件：old_str为空则创建文件；否则替换首次出现的old_str",
            "parameters": {
                "type": "object",
                "properties": {
                    "path": {"type": "string", "description": "文件路径"},
                    "old_str": {"type": "string", "description": "要替换的文本，空字符串=创建新文件"},
                    "new_str": {"type": "string", "description": "替换后的文本"}
                },
                "required": ["path", "old_str", "new_str"]
            }
        }
    }
]

# 系统提示词
SYSTEM_PROMPT = """你是一个编程助手智能体。你可以使用以下工具：

  • read_file：读取文件的完整内容
  • list_files：列出目录中的文件和子目录
  • edit_file：编辑文件（old_str为空则创建，非空则替换）

使用规则：
1. 先判断是否需要工具
2. 需要时直接发起工具调用
3. 收到结果后继续推理或回复
4. 创建文件时 old_str 传空字符串""
5. 修改文件前，先 read_file 了解当前内容
"""

# ── SimpleAgent 类 ────────────────────────────
class SimpleAgent:
    """最小 AI 编程智能体"""
    
    def __init__(self, api_key, base_url, model):
        self.client = OpenAI(api_key=api_key, base_url=base_url)
        self.model = model
        self.conversation = [{"role": "system", "content": SYSTEM_PROMPT}]
    
    def chat(self, user_message):
        """处理一条用户消息，运行完整的 Agent Loop"""
        print(f"📝 用户: {user_message}")
        self.conversation.append({"role": "user", "content": user_message})
        
        while True:  # ← 内层循环
            response = self.client.chat.completions.create(
                model=self.model,
                messages=self.conversation,
                tools=TOOLS,
                tool_choice="auto",
            )
            msg = response.choices[0].message
            
            # 没有 tool_calls → LLM 直接回复，结束内层循环
            if not msg.tool_calls:
                self.conversation.append({"role": "assistant", "content": msg.content})
                print(f"🤖 Agent: {msg.content or '(空消息)'}\n")
                return msg.content
            
            # 有 tool_calls → 执行工具
            self.conversation.append({
                "role": "assistant",
                "content": msg.content,
                "tool_calls": [{
                    "id": tc.id,
                    "type": "function",
                    "function": {"name": tc.function.name, "arguments": tc.function.arguments}
                } for tc in msg.tool_calls]
            })
            
            # 逐个执行工具调用
            for tc in msg.tool_calls:
                tool_name = tc.function.name
                tool_args = json.loads(tc.function.arguments)
                tool_fn = TOOL_REGISTRY.get(tool_name)
                
                print(f"  🔧 调用工具: {tool_name}({json.dumps(tool_args, ensure_ascii=False)})")
                
                if tool_fn is None:
                    result = {"error": f"未知工具: {tool_name}"}
                else:
                    try:
                        result = tool_fn(**tool_args)
                    except Exception as e:
                        result = {"error": str(e)}
                
                preview = json.dumps(result, ensure_ascii=False)
                print(f"  📋 工具结果: {preview[:120]}{'...' if len(preview) > 120 else ''}")
                
                self.conversation.append({
                    "role": "tool",
                    "tool_call_id": tc.id,
                    "content": json.dumps(result, ensure_ascii=False)
                })
            # 继续内层循环，让 LLM 处理工具结果

# 初始化 Agent
agent = SimpleAgent(API_KEY, BASE_URL, MODEL)
print("✅ Agent 初始化完成！试试下面的示例吧 👇")

✅ Agent 初始化完成！试试下面的示例吧 👇


### 示例 1：创建文件

让 Agent 创建一个 `hello.py`。注意观察 `edit_file` 工具被调用时 `old_str` 为空。

In [4]:
agent.chat("创建一个文件 hello.py，内容为打印 'hello ,小帅' 的 Python 代码")

📝 用户: 创建一个文件 hello.py，内容为打印 'hello ,小帅' 的 Python 代码
  🔧 调用工具: edit_file({"path": "hello.py", "old_str": "", "new_str": "print('hello ,小帅')"})
  📋 工具结果: {"path": "D:\\00_mywork\\15_agent_loop_cc\\hello.py", "action": "created_file"}
🤖 Agent: 
已成功创建 hello.py 文件，文件内容为：
```python
print('hello ,小帅')
```

这个文件包含了您要求的 Python 代码，运行后会打印出 "hello ,小帅"。



'\n已成功创建 hello.py 文件，文件内容为：\n```python\nprint(\'hello ,小帅\')\n```\n\n这个文件包含了您要求的 Python 代码，运行后会打印出 "hello ,小帅"。'

### 示例 2：修改文件（重点观察！）

让 Agent 给 `hello.py` 添加一个函数。**仔细观察输出**：Agent 会**自动先读文件，再修改**——这个「先读后改」的决策完全是 LLM 自己做出的！

In [7]:
agent.chat("给 hello.py 添加一个加法函数 add(a, b)，返回两数之和")

📝 用户: 给 hello.py 添加一个加法函数 add(a, b)，返回两数之和
  🔧 调用工具: read_file({"path": "hello.py"})
  📋 工具结果: {"file_path": "D:\\00_mywork\\15_agent_loop_cc\\hello.py", "content": "print('hello ,小帅')"}
  🔧 调用工具: edit_file({"path": "hello.py", "old_str": "print('hello ,小帅')", "new_str": "def add(a, b):\n    return a + b\n\nprint('hello ,小帅')"})
  📋 工具结果: {"path": "D:\\00_mywork\\15_agent_loop_cc\\hello.py", "action": "edited"}
  🔧 调用工具: read_file({"path": "hello.py"})
  📋 工具结果: {"file_path": "D:\\00_mywork\\15_agent_loop_cc\\hello.py", "content": "def add(a, b):\n    return a + b\n\nprint('hello ...
🤖 Agent: 
已成功为 hello.py 添加了 add(a, b) 函数。现在文件内容为：

```python
def add(a, b):
    return a + b

print('hello ,小帅')
```

这个 add 函数可以接收两个参数 a 和 b，并返回它们的和。



"\n已成功为 hello.py 添加了 add(a, b) 函数。现在文件内容为：\n\n```python\ndef add(a, b):\n    return a + b\n\nprint('hello ,小帅')\n```\n\n这个 add 函数可以接收两个参数 a 和 b，并返回它们的和。"

👀 **查看左侧文件树**，双击 `hello.py` 看看内容是否已经更新！

### 示例 3：列出目录

让 Agent 看看当前目录有哪些文件。

In [5]:
agent.chat("列出当前目录的所有文件")

📝 用户: 列出当前目录的所有文件
  🔧 调用工具: list_files({"path": "."})
  📋 工具结果: {"path": "D:\\00_mywork\\15_agent_loop_cc", "files": [{"filename": ".env", "type": "file"}, {"filename": ".env.example",...
  🔧 调用工具: read_file({"path": "hello.py"})
  📋 工具结果: {"error": "[Errno 2] No such file or directory: 'D:\\\\00_mywork\\\\15_agent_loop_cc\\\\hello.py'"}
  🔧 调用工具: edit_file({"path": "hello.py", "old_str": "", "new_str": "print('hello ,小帅')"})
  📋 工具结果: {"path": "D:\\00_mywork\\15_agent_loop_cc\\hello.py", "action": "created_file"}
  🔧 调用工具: list_files({"path": "."})
  📋 工具结果: {"path": "D:\\00_mywork\\15_agent_loop_cc", "files": [{"filename": ".env", "type": "file"}, {"filename": ".env.example",...
🤖 Agent: 
现在可以看到当前目录包含以下文件和目录：

**文件：**
- .env
- .env.example
- agent.ipynb
- CLAUDE.md
- hello.py （刚刚创建的）

**目录：**
- .ipynb_checkpoints
- .pi
- openspec
- test
- work

hello.py 文件已经成功创建并出现在目录列表中。



'\n现在可以看到当前目录包含以下文件和目录：\n\n**文件：**\n- .env\n- .env.example\n- agent.ipynb\n- CLAUDE.md\n- hello.py （刚刚创建的）\n\n**目录：**\n- .ipynb_checkpoints\n- .pi\n- openspec\n- test\n- work\n\nhello.py 文件已经成功创建并出现在目录列表中。'

### 示例 4：异常处理

故意让 Agent 读取一个不存在的文件，观察它如何处理错误。

In [9]:
agent.chat("读取文件 not_exist_xyz.py 的内容")

📝 用户: 读取文件 not_exist_xyz.py 的内容
  🔧 调用工具: read_file({"path": "not_exist_xyz.py"})
  📋 工具结果: {"error": "[Errno 2] No such file or directory: 'not_exist_xyz.py'"}
🤖 Agent: 
文件 `not_exist_xyz.py` 不存在，系统返回了错误信息：`[Errno 2] No such file or directory: 'not_exist_xyz.py'`

这个文件在当前目录中不存在。



"\n文件 `not_exist_xyz.py` 不存在，系统返回了错误信息：`[Errno 2] No such file or directory: 'not_exist_xyz.py'`\n\n这个文件在当前目录中不存在。"

---

## 你已经体验了 Agent 的核心能力！

总结一下刚才发生了什么：

| 示例 | Agent 做了什么 | 关键观察 |
|------|---------------|---------|
| 创建文件 | 调用 `edit_file`，`old_str` 为空 | 单次工具调用完成 |
| 修改文件 | **先** `read_file`，**再** `edit_file` | LLM 自主决定了两步！ |
| 列出目录 | 调用 `list_files` | 返回结构化文件列表 |
| 异常处理 | `read_file` 失败 → 错误传回 LLM → 友好回复 | Agent 能优雅处理错误 |

> 接下来，我们逐段拆解代码，看看这个「魔法」到底是怎么实现的。

## Part 3：逐段拆解 — 魔法是怎么发生的

刚才你看到 Agent 能自主创建文件、修改代码、处理错误。现在我们把代码**拆成一块一块**，理解每一部分在做什么。

---

### 3.1 配置模型

这是整个 Agent 的「发动机」。任何支持 OpenAI 兼容接口的大模型都可以用。

**用生活中的概念来理解：**

| 概念 | 生活类比 | 代码中的体现 |
|------|---------|-------------|
| `api_key` | 门禁卡，证明你有权限使用服务 | `API_KEY` |
| `base_url` | 服务器地址，告诉程序去哪找模型 | `BASE_URL` |
| `model` | 具体用哪个模型 | `MODEL` |

In [ ]:
# 创建 OpenAI 客户端（连接到你自己的服务商）
from openai import OpenAI

client = OpenAI(
    api_key=API_KEY,       # 你的「门禁卡」
    base_url=BASE_URL       # 你的服务商地址
)

# 验证一下
r = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "回复 OK"}],
    max_tokens=10
)
print(f"✅ 模型 {r.model} 连接正常")
print(f"   回复：{r.choices[0].message.content}")

### 3.2 定义工具 — Agent 的「手和眼睛」

LLM 本身只是一个文本生成器，它**不能操作文件**。所谓「AI 编程助手」，其实是：LLM 告诉你它想做什么 → 你的代码帮它做 → 把结果告诉 LLM。

这三个工具就是 Agent 的「手和眼睛」。

#### 工具 1：`read_file` — 眼睛

读取文件内容，让 LLM「看到」代码。

In [10]:
def read_file(path: str) -> dict:
    """读取文件的完整内容"""
    from pathlib import Path
    full_path = Path(path).expanduser().resolve()
    content = full_path.read_text(encoding="utf-8")
    return {"file_path": str(full_path), "content": content}

# 试一试
result = read_file("hello.py")
print(f"文件路径：{result['file_path']}")
print(f"文件内容：\n{result['content']}")

文件路径：D:\00_mywork\15_agent_loop_cc\hello.py
文件内容：
def add(a, b):
    return a + b

print('hello ,小帅')


#### 工具 2：`list_files` — 导航

列出目录内容，让 LLM「浏览」项目结构。

In [11]:
def list_files(path: str) -> dict:
    """列出目录中的文件和子目录"""
    from pathlib import Path
    full_path = Path(path).expanduser().resolve()
    items = []
    for item in sorted(full_path.iterdir()):
        items.append({
            "filename": item.name,
            "type": "file" if item.is_file() else "dir"
        })
    return {"path": str(full_path), "files": items}

# 试一试
result = list_files(".")
for f in result["files"]:
    print(f"  {'📄' if f['type']=='file' else '📁'} {f['filename']}")

  📄 .env
  📄 .env.example
  📁 .ipynb_checkpoints
  📁 .pi
  📄 agent.ipynb
  📄 agent.py
  📄 build_notebook.py
  📄 cells_data.json
  📄 CLAUDE.md
  📄 hello.py
  📁 nul
  📁 openspec
  📄 prompt_his.md
  📄 README.md
  📄 requirements.txt
  📁 test
  📄 test_agent.py


#### 工具 3：`edit_file` — 手

最复杂的工具，有**两种模式**：

| 条件 | 行为 | 使用场景 |
|------|------|---------|
| `old_str` 为空 `""` | 创建/覆盖文件 | 新建文件 |
| `old_str` 非空 | 替换文件中首次出现的匹配文本 | 修改现有文件 |

In [ ]:
def edit_file(path: str, old_str: str, new_str: str) -> dict:
    """编辑文件：old_str为空则创建，非空则替换"""
    from pathlib import Path
    full_path = Path(path).expanduser().resolve()

    if old_str == "":
        # 模式 1：创建新文件
        full_path.write_text(new_str, encoding="utf-8")
        return {"path": str(full_path), "action": "created_file"}

    # 模式 2：替换文本
    original = full_path.read_text(encoding="utf-8")
    if old_str not in original:
        return {"path": str(full_path), "action": "old_str not found"}

    edited = original.replace(old_str, new_str, 1)
    full_path.write_text(edited, encoding="utf-8")
    return {"path": str(full_path), "action": "edited"}

# 试一试：模式 1 创建文件
result = edit_file("test_temp.txt", "", "这是测试内容")
print(f"创建文件：{result}")

# 试一试：模式 2 替换文本
result = edit_file("test_temp.txt", "测试", "演示")
print(f"替换文本：{result}")

### 3.3 工具注册 — 告诉 LLM「你能用什么」

现在有了三个工具函数，还需要**两样东西**把工具「介绍」给 LLM：

1. **`TOOL_REGISTRY`**：Python 字典，名称 → 函数映射（供代码查找和执行）
2. **`TOOLS`**：JSON 数组，描述每个工具的名称、用途、参数（供 LLM 理解）

`TOOLS` 是 LLM 的「菜单」，LLM 看了就知道有哪些工具可用、每个工具需要什么参数。

In [ ]:
# 工具注册表：名称 → 函数
TOOL_REGISTRY = {
    "read_file": read_file,
    "list_files": list_files,
    "edit_file": edit_file,
}

# 工具的 JSON Schema（LLM 的「菜单」）
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "read_file",
            "description": "读取文件的完整内容",
            "parameters": {
                "type": "object",
                "properties": {"path": {"type": "string"}},
                "required": ["path"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "list_files",
            "description": "列出目录中的文件和子目录",
            "parameters": {
                "type": "object",
                "properties": {"path": {"type": "string"}},
                "required": ["path"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "edit_file",
            "description": "编辑文件：old_str为空则创建，非空则替换",
            "parameters": {
                "type": "object",
                "properties": {
                    "path": {"type": "string"},
                    "old_str": {"type": "string"},
                    "new_str": {"type": "string"}
                },
                "required": ["path", "old_str", "new_str"]
            }
        }
    }
]

print("✅ 工具注册完成！")
print(f"   已注册 {len(TOOLS)} 个工具：", [t['function']['name'] for t in TOOLS])

### 3.4 系统提示词 — 给 LLM 的「说明书」

系统提示词是给 LLM 的角色设定和操作规则。它告诉 LLM：

- **你是谁**：编程助手
- **你能用什么**：三个工具
- **怎么用**：先判断、再调用、有结果后继续

系统提示词附加在对话历史的**最前面**，LLM 在整个对话中都会遵守这些规则。

In [ ]:
SYSTEM_PROMPT = """你是一个编程助手智能体。你可以使用以下工具：

  • read_file：读取文件的完整内容
  • list_files：列出目录中的文件和子目录
  • edit_file：编辑文件（old_str为空则创建，非空则替换）

使用规则：
1. 先判断是否需要工具
2. 需要时直接发起工具调用
3. 收到结果后继续推理或回复
4. 创建文件时 old_str 传空字符串""
5. 修改文件前，先 read_file 了解当前内容
"""

print("系统提示词：")
print(SYSTEM_PROMPT)

### 3.5 Agent Loop — 核心中的核心 ⭐

这是整个智能体的「心脏」。理解了这个循环，就理解了所有 AI 编程助手的工作原理。

```
用户输入："添加一个乘法函数"
     │
     ▼
┌──────────────┐     ┌────────────────┐
│  第 1 轮      │────▶│ read_file()    │
│  LLM 推理    │     │ 返回文件内容     │
└──────┬───────┘     └───────┬────────┘
       │                     │
       │         结果送回 LLM │
       ▼                     │
┌──────────────┐     ┌──────▼─────────┐
│  第 2 轮      │────▶│ edit_file()    │
│  LLM 推理    │     │ 文件修改成功     │
└──────┬───────┘     └───────┬────────┘
       │                     │
       │         结果送回 LLM │
       ▼                     │
┌──────────────┐              │
│  第 3 轮      │  无 tool_calls
│  LLM 推理    │──────────────┘
└──────┬───────┘
       │
       ▼
"乘法函数已添加完成！"  ← 内层循环结束

回到外层循环，等待下一个用户输入
```

**关键理解**：
- 每一轮 LLM 调用后，检查「是否还需要工具？」
- 如果需要 → 执行工具 → 结果送回 → 继续内层循环
- 如果不需要 → 输出回复 → 内层循环结束 → 等待用户下一条消息

In [ ]:
class SimpleAgent:
    """最小 AI 编程智能体"""
    
    def __init__(self, api_key, base_url, model):
        self.client = OpenAI(api_key=api_key, base_url=base_url)
        self.model = model
        # 对话历史：以系统提示词开头
        self.conversation = [{"role": "system", "content": SYSTEM_PROMPT}]
    
    def chat(self, user_message):
        """处理一条用户消息，运行完整的 Agent Loop"""
        self.conversation.append({"role": "user", "content": user_message})
        
        while True:  # ← 内层循环：LLM 推理 → 工具执行 → 结果回传
            response = self.client.chat.completions.create(
                model=self.model,
                messages=self.conversation,
                tools=TOOLS,
                tool_choice="auto",
            )
            msg = response.choices[0].message
            
            # ── 关键判断：LLM 要调用工具吗？ ──
            if not msg.tool_calls:
                # 不需要工具 → 直接回复用户 → 结束内层循环
                self.conversation.append({"role": "assistant", "content": msg.content})
                return msg.content
            
            # 需要工具 → 追加 assistant 消息（含 tool_calls）
            self.conversation.append({
                "role": "assistant",
                "content": msg.content,
                "tool_calls": [{
                    "id": tc.id, "type": "function",
                    "function": {"name": tc.function.name, "arguments": tc.function.arguments}
                } for tc in msg.tool_calls]
            })
            
            # 执行每个工具
            for tc in msg.tool_calls:
                tool_name = tc.function.name
                tool_args = json.loads(tc.function.arguments)
                tool_fn = TOOL_REGISTRY.get(tool_name)
                
                try:
                    result = tool_fn(**tool_args)
                except Exception as e:
                    result = {"error": str(e)}
                
                # 结果追加为 tool 角色消息（含 tool_call_id 匹配）
                self.conversation.append({
                    "role": "tool",
                    "tool_call_id": tc.id,
                    "content": json.dumps(result, ensure_ascii=False)
                })
            # 循环继续，让 LLM 处理工具结果

print("✅ SimpleAgent 类已定义")
print("   核心就一个 chat() 方法 + 一个 while True 内层循环")

### Part 3 小结

恭喜！你已经理解了 Agent Loop 的全部组成部分：

| 组件 | 作用 | 代码量 |
|------|------|--------|
| 模型配置 | 连接 LLM（发动机） | ~5 行 |
| 三个工具 | Agent 的「手和眼睛」 | ~30 行 |
| 工具注册 | 告诉 LLM 能做什么（菜单） | ~40 行 |
| 系统提示词 | LLM 的角色说明书 | ~10 行 |
| **Agent Loop** | **核心循环：推理→执行→回传** | **~30 行** |

**核心公式：**
> LLM 决定要什么工具 → 代码执行 → 结果送回 → 继续循环 → 直到 LLM 不再要工具

---

## Part 4：🎉 自由探索

现在你已经理解了原理——试试你自己的任务吧！

Agent 保留了之前的对话上下文（还记得 hello.py），你可以继续操作，也可以做全新的任务。

In [ ]:
# 🎉 自由探索模式
# 输入任何编程任务，观察 Agent 的工作过程
# 输入 'exit' 退出

while True:
    task = input("\n👉 你的任务（输入 exit 退出）: ").strip()
    if task.lower() == "exit" or task == "":
        break
    agent.chat(task)

print("👋 探索结束！翻到下一个 Cell 查看总结。")

## 📝 总结：你学到了什么

### Agent Loop = 一个循环

```python
while True:                          # 对每条用户消息
    response = LLM(messages, tools)  #   调用 LLM
    if 没有 tool_calls:              #   不需要工具？
        print(response)              #     直接回复
        break                        #     等下一句
    执行所有工具调用()                #   需要工具 → 执行
    结果追加到 messages               #   结果送回去
                                     #   再问一次 LLM
```

### 为什么这个模式如此强大？

1. **LLM 自主决策**：代码没有写死「修改前先读文件」，LLM 自己决定的
2. **工具可扩展**：加新工具只需定义函数 + 注册，LLM 自动学会使用
3. **错误自恢复**：工具执行失败 → 错误回传 → LLM 调整策略
4. **上下文连续**：对话历史累积，LLM 记住之前的所有操作

### 这 150 行代码 = Claude / Cursor / Copilot 的核心

生产级 AI 编程助手多出来的：
- 更多工具（bash、grep、web search...）
- 更好的错误处理（模糊匹配、自动重试...）
- 流式输出、UI 界面...
- 上下文管理（长对话的裁剪和摘要...）

但 **核心循环一模一样**。

### 延伸学习

- 📄 [原文: The Emperor Has No Clothes](https://www.mihaileric.com/The-Emperor-Has-No-Clothes/)
- 🧪 [GitHub: 完整源码（~200行）](https://shorturl.at/HmMeI)
- 🔧 练习：试试给 Agent 添加一个新工具（比如 `grep` 搜索代码）